#REQUIRED: Getting Data for Your Project
***First, save a copy of this journal so you can edit it!***

Running this journal should get you and your team started with gathering brain data for your project! To summarize what this journal template does:
*   Walks through and imports the libraries and packages needed.
*   Gets you an API key to access FlyWire and the fruit fly brain data.
*   Shows a quick example on accessing data from FlyWire using *fafbseg*.
*   Gives information on exporting data to .json along with some quick tips.

If you and your team need help with anything regarding the journal, reach out to an organizer who will either be in the very front of the room or walking around! We're happy to help with anything that may come up. 🪰 🧠

We also left some comments in text blocks to help you better understand what is going on. As long as you use *fafbseg* and FlyWire, you are welcome to do whatever you want! Get creative, and have fun!


---





This first block of code below will import all the libraries and packages necessary for running fafbseg. See the breakdown below, and feel free to include other libraries and packages if they'll help with analysis or process the information you want better.

*   **fafbseg:** the library that contains the fruit fly brain we'll be using for this Hack Day
*   **navis:** Neuron Analysis and Visualization System designed for processing, analyzing, and interactively visualizing brain data
*   **plotly:** open-source graphing library to create interactive, web-based 2D and 3D visualizations.
*   **crackle-codec:** high-performance compression library so our code using *fafbseg* runs faster

\
**IMPORTANT: USE FAFBSEG AND FLYWIRE TO GET YOUR FRUIT FLY BRAIN DATA! WE WILL BE CHECKING FOR THIS FOR YOUR SUBMISSION TO QUALIFY!**

In [ ]:
!pip install fafbseg navis plotly crackle-codec



---


The second code block will set your API key for this journal. Two cases to keep in mind:
1.   **If you are getting an API key for this Hack Day for the first time,** please visit the embedded link [here](https://global.daf-apis.com/auth/api/v1/create_token). This may prompt you to sign-in using Google. This will generate your API key.
2.   **If you already generated an API key for this Hack Day** and you may have forgot it or lost it from your clipboard, please visit this link [here](https://global.daf-apis.com/auth/api/v1/user/token). This will display your API key so you can copy-paste it again.

Once you copy your API key, paste it where you see "YOUR API KEY GOES HERE" as a String. Running the block and getting "Token succesfully stored." (yes with the typo) means you are good to go!

\
Running this block multiple times after its first success may result in an error like this popping up:
```
ValueError: Key "token" already exists in token file "/root/.cloudvolume/secrets/global.daf.apis.com-cave-secret.json"
```
This is because the API key was already set after the first successful run. Restarting your Colab session will not affect this error, but if you got this error, then you are already set to move on!



In [ ]:
from fafbseg import flywire

flywire.set_chunkedgraph_secret("my api key WAS here")



---

Assuming there were no errors in the first two code blocks, this last code block should work flawlessly! All this code block is doing is testing if you are running the library properly. It should output a few print statements (ignore warnings if it says anything about "appearing to have ## somas", it is normal and did that to us while testing). It should also output a "3D antomical skeleton of the left side of a fruit fly brain". It's also interactive! Yummy!

In [ ]:
from fafbseg import flywire
import navis

flywire.set_default_dataset("public")

NC = flywire.NeuronCriteria
matches = flywire.search_annotations(NC(type="DA1_lPN", side="left"))
print(f"Found {len(matches)} neurons. Sample Root IDs:")

print(matches[["root_id", "cell_type", "side"]].head())

sample_root_id = matches["root_id"].iloc[0]
skeleton = flywire.get_skeletons(sample_root_id)

navis.plot3d(skeleton)

# Next Steps
Running the entire section above should get you access to the data and libraries you'll need for your project! If everything worked, continue reading here. Otherwise, please get the attention of an organizer and see if you can troubleshoot the issues.

From here, your project is entirely up to you and your team! We recommend continuing to use this Google Colab journal (make a copy of this if you haven't already!) to get whatever data you'll need for your project. You can save this data in an image using *navis* or a .json file and upload this to your GitHub repo for use in your project.

Here's a quick example on exporting your data. In this example, we want to save the adjacency matrix from the synapses, so:
```
conn = flywire.synapses.get_adjacency_matrix(
  upstream=[720575940625431866]
  downstream=[720575940822603417]
)

conn.to_json("brain_network.json")

# Optional code below. You can also go to the Folder icon on the left, find your file, and download it to your computer.
# This will make Colab automatically download your file when the block is ran.
from google.colab import files

files.download("brain_network.json")
```



After getting your data in a GitHub repo, you should be able to utilize the data in your project's scripts. When it's your team's time to demo your project, we'll ask for a quick check on how/where you got your data. **Show the file you ported from here or some evidence of data being collected from this Colab to qualify for this Hack Day's judging!**

*Brief reminder: it may be unwise to leave your API key visible in your public repo. Double-check your submissions to ensure a) your GitHub repo link is public, and b) your API keys are not publicly available to use.*

# Your Project
Start a new code block to begin gathering your data (or use an existing code block, whichever you prefer).

In [ ]:
# Export the neuron rendering as transparent sprites.
import navis
from fafbseg import flywire

flywire.set_default_dataset("public")

# Fetch target neurons (e.g., giant fiber / motor neurons)
neurons = flywire.get_skeletons([720575940625431866])

# Render to 2D image with transparent background
fig, ax = navis.plot2d(neurons, color="green", method="2d")
ax.axis("off")
fig.patch.set_alpha(0.0)

# Save image asset for local GUI
fig.savefig("fly_brain_sprite.png", transparent=True, bbox_inches="tight")

NC = flywire.NeuronCriteria

# 1. Dynamically fetch valid source neuron IDs from the active public release
sources = flywire.search_annotations(NC(cell_type="DA1_lPN"))["root_id"].tolist()

# 2. Find their actual top downstream partners automatically
connectivity = flywire.get_connectivity(sources)
downstream = connectivity[connectivity["pre"].isin(sources)]
top_targets = (
    downstream.sort_values("weight", ascending=False)["post"]
    .unique()[:5]
    .tolist()
)

# 3. Build an adjacency matrix guaranteed to have valid IDs and real synaptic weights
conn = flywire.get_adjacency(sources=sources, targets=top_targets)
conn.to_json("brain_network.json")

In [ ]:
import json
from fafbseg import flywire

flywire.set_default_dataset("public")
NC = flywire.NeuronCriteria

# 1. Blue/UV color pathway (Dm8 -> Tm5c in the medulla)
dm8_ids = flywire.search_annotations(NC(type="Dm8"))["root_id"].tolist()
tm5c_ids = flywire.search_annotations(NC(type="Tm5c"))["root_id"].tolist()
blue_conn = flywire.get_adjacency(sources=dm8_ids, targets=tm5c_ids)
blue_synapses = int(blue_conn.values.sum())

# 2. Dark-object / small-target pathway (LC11 -> downstream targets)
lc11_ids = flywire.search_annotations(NC(type="LC11"))["root_id"].tolist()
lc11_conn = flywire.get_connectivity(lc11_ids)
dark_synapses = int(lc11_conn[lc11_conn["pre"].isin(lc11_ids)]["weight"].sum())

with open("vision_attraction.json", "w") as f:
    json.dump({
        "blue_circuit": "Dm8 -> Tm5c",
        "blue_synapses": blue_synapses,
        "dark_circuit": "LC11 output",
        "dark_synapses": dark_synapses
    }, f)

print(f"Exported! Blue pathway: {blue_synapses} synapses | Dark-object pathway: {dark_synapses} synapses")

Actual Updated Connectome Exporter so it stops using random for everything.

In [ ]:
import json
import numpy as np
import pandas as pd
from fafbseg import flywire

flywire.set_default_dataset("public")
NC = flywire.NeuronCriteria


def get_lr_ids(cell_types):
    """Fetch root IDs split by Left and Right brain hemispheres."""
    left_df = flywire.search_annotations(NC(type=cell_types, side="left"))
    right_df = flywire.search_annotations(NC(type=cell_types, side="right"))
    return left_df["root_id"].tolist(), right_df["root_id"].tolist()


def effective_pathway_weight(src_ids, dst_ids, two_hop_scale=0.15):
    """
    Compute total effective synaptic weight from src_ids -> dst_ids,
    combining direct 1-hop synapses and 2-hop interneuron pathways.
    """
    if not src_ids or not dst_ids:
        return 0.0

    # 1. Direct 1-hop synapses
    conn = flywire.get_connectivity(src_ids)
    out_edges = conn[conn["pre"].isin(src_ids)]
    direct_weight = float(out_edges[out_edges["post"].isin(dst_ids)]["weight"].sum())

    # 2. 2-hop synapses via top downstream interneurons (e.g., optic lobe -> DNa02)
    inter_df = (
        out_edges[~out_edges["post"].isin(dst_ids)]
        .groupby("post")["weight"]
        .sum()
        .sort_values(ascending=False)
        .head(40)
    )
    if inter_df.empty:
        return direct_weight

    inter_ids = inter_df.index.tolist()
    hop2_conn = flywire.get_connectivity(inter_ids)
    hop2_edges = hop2_conn[
        (hop2_conn["pre"].isin(inter_ids)) & (hop2_conn["post"].isin(dst_ids))
    ]

    if hop2_edges.empty:
        return direct_weight

    # Weight each 2-hop path by min(hop1, hop2) bottleneck strength
    merged = hop2_edges.merge(
        inter_df.rename("w1"), left_on="pre", right_index=True
    )
    two_hop_weight = float(np.minimum(merged["w1"], merged["weight"]).sum())

    return direct_weight + (two_hop_scale * two_hop_weight)


# --- 1. Query Left/Right Neuron Populations from FlyWire ---
loom_L, loom_R = get_lr_ids(["LPLC2", "LC4"])       # Visual looming detectors
cx_L, cx_R = get_lr_ids(["PFL3"])                  # Central Complex steering neurons
gf_L, gf_R = get_lr_ids(["DNp01"])                 # Giant Fiber escape motor neurons
steer_L, steer_R = get_lr_ids(["DNa02", "DNa01"])  # Walking/turning steering neurons

print(f"Loaded neurons -> Looming (L/R): {len(loom_L)}/{len(loom_R)}, "
      f"CX PFL3 (L/R): {len(cx_L)}/{len(cx_R)}, "
      f"Giant Fiber (L/R): {len(gf_L)}/{len(gf_R)}, "
      f"Steering DNs (L/R): {len(steer_L)}/{len(steer_R)}")

# --- 2. Build 2x2 Hemispheric Weight Matrices ---
# Rows = [Left Input, Right Input], Cols = [Left Motor, Right Motor]
W_loom_to_gf = np.array([
    [effective_pathway_weight(loom_L, gf_L), effective_pathway_weight(loom_L, gf_R)],
    [effective_pathway_weight(loom_R, gf_L), effective_pathway_weight(loom_R, gf_R)],
])

W_loom_to_steer = np.array([
    [effective_pathway_weight(loom_L, steer_L), effective_pathway_weight(loom_L, steer_R)],
    [effective_pathway_weight(loom_R, steer_L), effective_pathway_weight(loom_R, steer_R)],
])

W_cx_to_steer = np.array([
    [effective_pathway_weight(cx_L, steer_L), effective_pathway_weight(cx_L, steer_R)],
    [effective_pathway_weight(cx_R, steer_L), effective_pathway_weight(cx_R, steer_R)],
])

# Normalize matrices so motor firing rates stay cleanly bounded in [0, 1+]
W_loom_to_gf_norm = (W_loom_to_gf / (W_loom_to_gf.max() or 1.0)).round(4)
W_loom_to_steer_norm = (W_loom_to_steer / (W_loom_to_steer.max() or 1.0)).round(4)
W_cx_to_steer_norm = (W_cx_to_steer / (W_cx_to_steer.max() or 1.0)).round(4)

brain_export = {
    "dataset": "FlyWire public",
    "raw_synapses": {
        "loom_to_gf": W_loom_to_gf.round(1).tolist(),
        "loom_to_steer": W_loom_to_steer.round(1).tolist(),
        "cx_to_steer": W_cx_to_steer.round(1).tolist(),
    },
    "weights": {
        "loom_to_gf": W_loom_to_gf_norm.tolist(),
        "loom_to_steer": W_loom_to_steer_norm.tolist(),
        "cx_to_steer": W_cx_to_steer_norm.tolist(),
    },
    "escape_threshold": 0.30,
}

with open("fly_brain_weights.json", "w") as f:
    json.dump(brain_export, f, indent=2)

print("\nExported fly_brain_weights.json!")
print("Looming -> Giant Fiber (raw):\n", W_loom_to_gf.round(1))
print("Looming -> Steering DNs (raw):\n", W_loom_to_steer.round(1))
print("CX PFL3 -> Steering DNs (raw):\n", W_cx_to_steer.round(1))